# Daily Facts volume pilot

This notebook tests the local Daily Facts 1.0.0 handoff using ES, NQ and GC
sessions from June 28, 2021 through September 18, 2026. It compares first-hour
and last-hour volume within each asset and session. Estimates describe the
retained population. Asset calendars and eligible sample sizes can differ.

In [ ]:
from datetime import date
import json
from pathlib import Path

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd

from abacus_bhishma.daily_facts_loader import load_daily_facts
from abacus_bhishma.paths import resolve_publication_root, workspace_root

workspace = workspace_root()
study_id = "31_daily_facts_volume_pilot"
cwd = Path.cwd()
staged = cwd.is_relative_to(workspace / "work" / study_id) and (cwd / "config.json").is_file()
lock_path = cwd / "inputs.lock.json" if staged else workspace / "studies" / study_id / "inputs.lock.json"
pin = json.loads(lock_path.read_text())
assets = pin["assets"]
columns = [
    "release_id", "asset", "session_date", "calendar_status",
    "globex_aggregate_ready", "rth_aggregate_ready", "rth_volume",
    "first60_volume", "last60_volume", "first60_volume_share", "last60_volume_share",
    "first60_aggregate_available_at", "last60_aggregate_available_at",
    "first60_volume_share_available_at", "last60_volume_share_available_at",
    "rth_start", "rth_end",
]
data = load_daily_facts(
    resolve_publication_root(), manifest_path=pin["manifest_path"],
    manifest_sha256=pin["manifest_sha256"], assets=tuple(assets),
    start=date.fromisoformat(pin["start"]), end_exclusive=date.fromisoformat(pin["end_exclusive"]),
    columns=columns, verify_selected_files=True,
).to_pandas()
for name in ("first60_volume", "last60_volume", "rth_volume"):
    data[name] = data[name].astype("Int64")

def save_table(frame, name):
    if staged:
        frame.to_csv(cwd / "tables" / name, index=False, float_format="%.17g")

## Coverage

How much of the retained roster can support a paired hour comparison?

### Which sessions support the comparison?

Count the full weekday roster, calendar exclusions, missing hour pairs, and
the subset with complete positive RTH volume. Each asset and session contributes
one observation. Globex completeness is recorded separately.

In [ ]:
if data.duplicated(["release_id", "asset", "session_date"]).any():
    raise ValueError("duplicate Daily Facts asset/session key")
if set(data["release_id"]) != {pin["dataset_id"]} or set(data["asset"]) != set(assets):
    raise ValueError("unexpected pinned release or asset identity")
if not (
    (data["first60_aggregate_available_at"] == data["rth_start"] + pd.Timedelta(hours=1))
    & (data["last60_aggregate_available_at"] == data["rth_end"])
    & (data["first60_volume_share_available_at"] == data["rth_end"])
    & (data["last60_volume_share_available_at"] == data["rth_end"])
).all():
    raise ValueError("hour or share availability differs from the declared RTH timing")

data["eligible"] = data["calendar_status"].eq("eligible")
data["paired"] = data["eligible"] & data["first60_volume"].notna() & data["last60_volume"].notna()
data["share_paired"] = (
    data["paired"] & data["rth_aggregate_ready"] & data["rth_volume"].gt(0).fillna(False)
)
data["rth_complete_globex_incomplete"] = (
    data["eligible"] & data["rth_aggregate_ready"] & ~data["globex_aggregate_ready"]
)
coverage = []
for asset in assets:
    group = data.loc[data["asset"].eq(asset)]
    eligible = int(group["eligible"].sum())
    paired = int(group["paired"].sum())
    share_paired = int(group["share_paired"].sum())
    coverage.append({
        "asset": asset, "weekday_roster": len(group), "calendar_excluded": len(group) - eligible,
        "eligible": eligible, "incomplete_hour_pair": eligible - paired,
        "paired": paired, "share_excluded_after_pairing": paired - share_paired,
        "share_paired": share_paired,
        "rth_complete_globex_incomplete": int(group["rth_complete_globex_incomplete"].sum()),
    })
coverage = pd.DataFrame(coverage)
save_table(coverage, "coverage.csv")
excluded = data.loc[~data["share_paired"], ["asset", "session_date", "calendar_status", "paired"]].copy()
excluded["reason"] = excluded["calendar_status"].where(
    excluded["calendar_status"].ne("eligible"),
    excluded["paired"].map({True: "incomplete_or_zero_rth_volume", False: "incomplete_hour_pair"}),
)
save_table(excluded.sort_values(["asset", "session_date"]), "exclusions.csv")

In [ ]:
gc = coverage.set_index("asset").loc["GC"]
coverage_findings = (
    f"The paired sample contains {coverage['paired'].sum():,} asset sessions. "
    f"GC has {gc.paired:,} paired sessions out of {gc.eligible:,} calendar-eligible dates. "
    f"The loader preserves {gc.rth_complete_globex_incomplete:,} complete GC RTH windows "
    "with incomplete Globex coverage."
)
display(Markdown(coverage_findings))

In [ ]:
display(coverage)

## Volume

How do the two RTH hours compare within each asset?

### How does first-hour volume compare with last-hour volume?

Pair first and last hour within the same eligible asset session. Count ties in
the denominator and report volume in contracts.

In [ ]:
hours = []
for asset in assets:
    paired = data.loc[data["asset"].eq(asset) & data["paired"]]
    first, last = paired["first60_volume"], paired["last60_volume"]
    hours.append({
        "asset": asset, "paired_sessions": len(paired),
        "mean_first_contracts": float(first.mean()), "mean_last_contracts": float(last.mean()),
        "median_first_contracts": float(first.median()), "median_last_contracts": float(last.median()),
        "median_first_minus_last_contracts": float((first - last).median()),
        "first_higher_sessions": int((first > last).sum()), "equal_sessions": int((first == last).sum()),
        "first_higher_fraction": float((first > last).mean()),
    })
hours = pd.DataFrame(hours)
save_table(hours, "paired_hours.csv")

In [ ]:
hours_findings = " ".join(
    f"{r.asset}: first-hour volume exceeds last-hour volume in "
    f"{r.first_higher_sessions:,} of {r.paired_sessions:,} pairs ({r.first_higher_fraction:.1%})."
    for r in hours.itertuples(index=False)
)
display(Markdown(hours_findings))

In [ ]:
display(hours.round({
    "mean_first_contracts": 1, "mean_last_contracts": 1, "first_higher_fraction": 4,
}))

### What fraction of RTH volume falls in each hour?

Use the same paired sessions for the two shares within each asset. This subset
also requires complete positive RTH volume. Weight each session equally. The
fractions become available at RTH close.

In [ ]:
shares = []
for asset in assets:
    usable = data.loc[data["asset"].eq(asset) & data["share_paired"]]
    shares.append({
        "asset": asset, "share_paired_sessions": len(usable),
        "mean_first_rth_share": float(usable["first60_volume_share"].mean()),
        "mean_last_rth_share": float(usable["last60_volume_share"].mean()),
    })
shares = pd.DataFrame(shares)
save_table(shares, "rth_shares.csv")

In [ ]:
share_findings = " ".join(
    f"{r.asset}: {r.mean_first_rth_share:.1%} in the first hour and "
    f"{r.mean_last_rth_share:.1%} in the last hour across {r.share_paired_sessions:,} sessions."
    for r in shares.itertuples(index=False)
)
display(Markdown(share_findings))

In [ ]:
display(shares.round({"mean_first_rth_share": 4, "mean_last_rth_share": 4}))
figure, ax = plt.subplots(figsize=(7, 4), layout="constrained")
locations = range(len(shares))
ax.bar([x - .18 for x in locations], shares["mean_first_rth_share"], .36,
       label="First hour", color="#216c81")
ax.bar([x + .18 for x in locations], shares["mean_last_rth_share"], .36,
       label="Last hour", color="#a9532c")
ax.set_xticks(list(locations), [f"{r.asset}  (n={r.share_paired_sessions:,})"
                             for r in shares.itertuples(index=False)])
ax.set_ylabel("Mean per-session share of RTH volume")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylim(0, max(shares[["mean_first_rth_share", "mean_last_rth_share"]].max()) * 1.22)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
if staged:
    figure.savefig(cwd / "figures" / "rth_hour_shares.png", dpi=150)
display(figure)
plt.close(figure)